In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Настройки отображения
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

print("Библиотеки загружены")

Библиотеки загружены


## Первичный осмотр данных

### Что загружено
- orders: 99 441 заказ
- customers: 99 441 запись (но 96 096 уникальных людей)
- order_items: 112 650 товаров в заказах
- payments: 103 886 оплат
- reviews: 99 224 отзыва

### Ключевые наблюдения
1. Большинство заказов (96 478) имеют статус delivered.
2. Есть пропуски в датах доставки — это нормально, часть заказов в процессе.
3. Оценки: большинство 5, но 11 424 отзыва с оценкой 1.
4. Главное: customer_id ≠ customer_unique_id. 
   Один человек может иметь несколько customer_id (по одному на заказ).
   Для анализа retention нужно использовать customer_unique_id.

In [6]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

files = os.listdir('../data/raw_data')
for f in sorted(files):
    print(f)

olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_orders_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


# вопрос №1: есть ли customer_unique_id?

In [7]:
orders = pd.read_csv('../data/raw_data/olist_orders_dataset.csv')
customers = pd.read_csv('../data/raw_data/olist_customers_dataset.csv')

print("Колонки в customers:")
print(customers.columns.tolist())
print()
print("Уникальных customer_id:", customers['customer_id'].nunique())
print("Уникальных customer_unique_id:", customers['customer_unique_id'].nunique())
print()

# 1. Оставляем только доставленные заказы
delivered_orders = orders[orders['order_status'] == 'delivered']

print("Доставленных заказов:", len(delivered_orders))

# 2. Соединяем заказы с клиентами, чтобы получить customer_unique_id
delivered_with_customers = delivered_orders.merge(
    customers[['customer_id', 'customer_unique_id']],
    on='customer_id',
    how='left')

# 3. Считаем уникальных клиентов
unique_customers_with_delivery = delivered_with_customers['customer_unique_id'].nunique()
print("Уникальных клиентов, получивших хотя бы один заказ:", unique_customers_with_delivery)

# 4. Сравниваем с общим числом клиентов
total_customers = customers['customer_unique_id'].nunique()
print(f"Всего уникальных клиентов в базе: {total_customers}")
print(f"Из них получили заказ: {unique_customers_with_delivery} ({unique_customers_with_delivery / total_customers:.1%})")

Колонки в customers:
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

Уникальных customer_id: 99441
Уникальных customer_unique_id: 96096

Доставленных заказов: 96478
Уникальных клиентов, получивших хотя бы один заказ: 93358
Всего уникальных клиентов в базе: 96096
Из них получили заказ: 93358 (97.2%)


# ~96 ТЫСЯЧ уникальных покупателей из ~99 тысяч

# вопрос №2: есть ли даты покупок?

In [8]:
print("Колонки в orders:")
print(orders.columns.tolist())
print()
print("Период данных:")
print("С:", orders['order_purchase_timestamp'].min())
print("По:", orders['order_purchase_timestamp'].max())
print()
print("Статусы заказов:")
print(orders['order_status'].value_counts())

Колонки в orders:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

Период данных:
С: 2016-09-04 21:15:19
По: 2018-10-17 17:30:18

Статусы заказов:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


# Для последующего анализа оставляем только доставленные заказы (96,478)

# вопрос №3: есть ли оценки?

In [9]:
reviews = pd.read_csv('../data/raw_data/olist_order_reviews_dataset.csv')

print("Колонки:", reviews.columns.tolist())
print()
print("Распределение оценок:")
print(reviews['review_score'].value_counts().sort_index())
print()
print(f"Средняя оценка: {reviews['review_score'].mean():.2f}")
print(f"Пропусков в review_score: {reviews['review_score'].isnull().sum()}")

Колонки: ['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

Распределение оценок:
review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

Средняя оценка: 4.09
Пропусков в review_score: 0


# Оценки есть, пропусков нет. Средняя — 4.09. Но 11 424 заказа с оценкой 1 — это 11.5% от всех. Гипотеза H2 (низкие оценки убивают возвращаемость) проверяема

# вопрос №4: есть ли данные о доставке?

In [10]:
print("Колонки с датами в orders:")
date_cols = [c for c in orders.columns if 'date' in c or 'timestamp' in c]
for c in date_cols:
    print(f"  {c}: пропусков {orders[c].isnull().sum()}")

Колонки с датами в orders:
  order_purchase_timestamp: пропусков 0
  order_delivered_carrier_date: пропусков 1783
  order_delivered_customer_date: пропусков 2965
  order_estimated_delivery_date: пропусков 0


# Есть и фактическая, и обещанная дата доставки. Значит, можно посчитать delivery_days и is_late. Гипотеза H1 проверяема. Пропуски в датах доставки — это недоставленные заказы, их отфильтруем

# вопрос №5: есть ли категории товаров?

In [11]:
products = pd.read_csv('../data/raw_data/olist_products_dataset.csv')
translation = pd.read_csv('../data/raw_data/product_category_name_translation.csv')

print("Колонки в products:", products.columns.tolist())
print()
print("Уникальных категорий (португальских):", products['product_category_name'].nunique())
print("Переводов:", len(translation))
print()
print("Примеры категорий:")
print(translation.head(10))

Колонки в products: ['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

Уникальных категорий (португальских): 73
Переводов: 71

Примеры категорий:
    product_category_name product_category_name_english
0            beleza_saude                 health_beauty
1  informatica_acessorios         computers_accessories
2              automotivo                          auto
3         cama_mesa_banho                bed_bath_table
4        moveis_decoracao               furniture_decor
5           esporte_lazer                sports_leisure
6              perfumaria                     perfumery
7   utilidades_domesticas                    housewares
8               telefonia                     telephony
9      relogios_presentes                 watches_gifts


# Категории есть — 73 штуки, есть перевод на английский. Гипотеза H3 проверяема. Но заметил: 73 категории против 71 перевода — две категории без перевода.

# вопрос №6: какого качества данные?

In [12]:
# Пропуски во всех таблицах
tables = {
    'orders': orders,
    'customers': customers,
    'reviews': reviews,
    'products': products,
}

for name, df in tables.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing) > 0:
        print(f"\n{name}:")
        print(missing)
    else:
        print(f"\n{name}: пропусков нет")


orders:
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

customers: пропусков нет

reviews:
review_comment_title      87656
review_comment_message    58247
dtype: int64

products:
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64


# все 2965 пропусков в order_delivered_customer_date — это заказы не в статусе delivered

In [13]:
print("Дубли в orders:", orders['order_id'].duplicated().sum())
print("Дубли в customers:", customers['customer_id'].duplicated().sum())
print("Дубли в reviews по order_id:", reviews['order_id'].duplicated().sum())

Дубли в orders: 0
Дубли в customers: 0
Дубли в reviews по order_id: 551


# В orders и customers дублей нет. А в reviews — 551 заказ с несколькими отзывами.

In [14]:
# Есть ли дата доставки раньше даты покупки?
anomaly = orders[
    orders['order_delivered_customer_date'] < orders['order_purchase_timestamp']
]
print("Заказов, где доставка раньше покупки:", len(anomaly))

Заказов, где доставка раньше покупки: 0


# 
# Этап 2. Подготовка к проверке гипотез
#

# H3 (категории) нужны products и translation, для H4 (штаты) — колонка customer_state из customers.

In [15]:
# Загружаем всё, что нужно для мастер-таблицы
orders = pd.read_csv('../data/raw_data/olist_orders_dataset.csv')
customers = pd.read_csv('../data/raw_data/olist_customers_dataset.csv')
order_items = pd.read_csv('../data/raw_data/olist_order_items_dataset.csv')
payments = pd.read_csv('../data/raw_data/olist_order_payments_dataset.csv')
reviews = pd.read_csv('../data/raw_data/olist_order_reviews_dataset.csv')
products = pd.read_csv('../data/raw_data/olist_products_dataset.csv')
translation = pd.read_csv('../data/raw_data/product_category_name_translation.csv')

print("Загружено:")
print(f"  orders: {orders.shape}")
print(f"  customers: {customers.shape}")
print(f"  order_items: {order_items.shape}")
print(f"  payments: {payments.shape}")
print(f"  reviews: {reviews.shape}")
print(f"  products: {products.shape}")
print(f"  translation: {translation.shape}")

Загружено:
  orders: (99441, 8)
  customers: (99441, 5)
  order_items: (112650, 7)
  payments: (103886, 5)
  reviews: (99224, 7)
  products: (32951, 9)
  translation: (71, 2)


# H1 Преобразование дат

In [16]:
date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

print("Даты преобразованы:")
print(orders[date_columns].dtypes)

Даты преобразованы:
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


Фильтруем заказы, убираем отменённые, потерянные покупки

In [17]:
print("До фильтрации:", len(orders))

orders_delivered = orders[orders['order_status'] == 'delivered'].copy()

print("После фильтрации:", len(orders_delivered))
print(f"Отфильтровано: {len(orders) - len(orders_delivered)} заказов")

До фильтрации: 99441
После фильтрации: 96478
Отфильтровано: 2963 заказов


In [18]:
print("Пропусков в order_delivered_customer_date:", 
      orders_delivered['order_delivered_customer_date'].isnull().sum())

Пропусков в order_delivered_customer_date: 8


# Найдена аномалия, среди доставленных заказов 8 не имеют даты доставки. Часть выборки небольшая - 0.008 %. Необходимо узнать причину отсутствия дат у этих заказов.

In [19]:
# Находим эти 8 заказов
anomaly = orders_delivered[orders_delivered['order_delivered_customer_date'].isnull()]

# Смотрим ключевые колонки
print(anomaly[[
    'order_id',
    'order_status',
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]])

                               order_id order_status order_purchase_timestamp   order_approved_at order_delivered_carrier_date order_delivered_customer_date order_estimated_delivery_date
3002   2d1e2d5bf4dc7227b3bfebb81328c15f    delivered      2017-11-28 17:44:07 2017-11-28 17:56:40          2017-11-30 18:12:23                           NaT                    2017-12-18
20618  f5dd62b788049ad9fc0526e3ad11a097    delivered      2018-06-20 06:58:43 2018-06-20 07:19:05          2018-06-25 08:05:00                           NaT                    2018-07-16
43834  2ebdfc4f15f23b91474edf87475f108e    delivered      2018-07-01 17:05:11 2018-07-01 17:15:12          2018-07-03 13:57:00                           NaT                    2018-07-30
79263  e69f75a717d64fc5ecdfae42b2e8e086    delivered      2018-07-01 22:05:55 2018-07-01 22:15:14          2018-07-03 13:57:00                           NaT                    2018-07-30
82868  0d3268bad9b086af767785e3f0fc0133    delivered      2018-07

# Можем проверить есть ли оценки у этих заказов, если они положительные,то причина отсутствия дат чисто техническая (не было записано)

In [20]:
anomaly_orders = anomaly['order_id'].tolist()
anomaly_reviews = reviews[reviews['order_id'].isin(anomaly_orders)]

print("Заказов с аномалией:", len(anomaly_orders))
print("Из них с отзывом:", len(anomaly_reviews))
print()
print(anomaly_reviews[['order_id', 'review_score']])

Заказов с аномалией: 8
Из них с отзывом: 8

                               order_id  review_score
10690  0d3268bad9b086af767785e3f0fc0133             5
24335  e69f75a717d64fc5ecdfae42b2e8e086             5
27655  f5dd62b788049ad9fc0526e3ad11a097             5
51996  2d858f451373b04fb5c984a1cc2defaf             5
56411  2ebdfc4f15f23b91474edf87475f108e             5
86011  2d1e2d5bf4dc7227b3bfebb81328c15f             5
88893  20edc82cf5400ce95e1afacc25798b31             5
92001  ab7c89dc1bf4a1ead9d6ec1ec8968a84             1


# Один заказ с отриц. оценкой может означать отсутствие доставленного заказа / товар не оправдал ожиданий.

In [21]:
# Удаляем заказы без даты доставки
orders_delivered = orders_delivered.dropna(
    subset=['order_delivered_customer_date']
).copy()

print("После удаления аномалий:", len(orders_delivered))
print("Пропусков в order_delivered_customer_date:", 
      orders_delivered['order_delivered_customer_date'].isnull().sum())

После удаления аномалий: 96470
Пропусков в order_delivered_customer_date: 0


# 8 заказов из 96 478 — это 0.008%. Они не повлияют на выводы. Удалить заказы с аномалией.

In [22]:
orders_delivered['delivery_days'] = (
    orders_delivered['order_delivered_customer_date'] -
    orders_delivered['order_purchase_timestamp']
).dt.days

orders_delivered['is_late'] = (
    orders_delivered['order_delivered_customer_date'] >
    orders_delivered['order_estimated_delivery_date']
)

print("Пропусков в delivery_days:", orders_delivered['delivery_days'].isnull().sum())
print("Доля опозданий:", f"{orders_delivered['is_late'].mean():.1%}")

Пропусков в delivery_days: 0
Доля опозданий: 8.1%


# данные о клиентах (для retention и h4)

In [23]:
orders_delivered = orders_delivered.merge(
    customers[['customer_id', 'customer_unique_id', 'customer_state']],
    on='customer_id',
    how='left'
)

print("Размер после merge:", orders_delivered.shape)
print("Пропусков в customer_unique_id:", 
      orders_delivered['customer_unique_id'].isnull().sum())
print("Пропусков в customer_state:", 
      orders_delivered['customer_state'].isnull().sum())

Размер после merge: (96470, 12)
Пропусков в customer_unique_id: 0
Пропусков в customer_state: 0


# выручку по каждому заказу

In [24]:
items_agg = order_items.groupby('order_id').agg(
    revenue=('price', 'sum'),
    freight=('freight_value', 'sum')
).reset_index()

print("Заказов с товарами:", len(items_agg))
print(items_agg.head())

Заказов с товарами: 98666
                           order_id  revenue  freight
0  00010242fe8c5a6d1ba2dd792cb16214    58.90    13.29
1  00018f77f2f0320c557190d7a144bdd3   239.90    19.93
2  000229ec398224ef6ca0657da4fc703e   199.00    17.87
3  00024acbcdf0a6daa1e931b038114c75    12.99    12.79
4  00042b26cf59d7ce69dfabb4e55b4fd9   199.90    18.14


revenue — сумма цен всех товаров в заказе.

freight — сумма доставки по всем товарам.

# сумма оплат

In [25]:
payments_agg = payments.groupby('order_id').agg(
    payment_value=('payment_value', 'sum')
).reset_index()

print("Заказов с оплатами:", len(payments_agg))
print(payments_agg.head())

Заказов с оплатами: 99440
                           order_id  payment_value
0  00010242fe8c5a6d1ba2dd792cb16214          72.19
1  00018f77f2f0320c557190d7a144bdd3         259.83
2  000229ec398224ef6ca0657da4fc703e         216.87
3  00024acbcdf0a6daa1e931b038114c75          25.78
4  00042b26cf59d7ce69dfabb4e55b4fd9         218.04


# дни доставки и флаг опоздания (H1)

In [26]:
orders_delivered['delivery_days'] = (
    orders_delivered['order_delivered_customer_date'] -
    orders_delivered['order_purchase_timestamp']
).dt.days

orders_delivered['is_late'] = (
    orders_delivered['order_delivered_customer_date'] >
    orders_delivered['order_estimated_delivery_date']
)

print(orders_delivered[['order_id', 'delivery_days', 'is_late']].head())
print()
print("Доля опозданий:", f"{orders_delivered['is_late'].mean():.1%}")

                           order_id  delivery_days  is_late
0  e481f51cbdc54678b7cc49136f2d6af7              8    False
1  53cdb2fc8bc7dce0b6741e2150273451             13    False
2  47770eb9100c2d0c44946d9cf07ec65d              9    False
3  949d5b44dbf5de918fe9c16f97b45f8a             13    False
4  ad21c59c0840e6cb83a9ceb5573f8159              2    False

Доля опозданий: 8.1%


In [27]:
# проверка аномалий
# Есть ли отрицательные дни доставки? (доставка раньше покупки)
anomaly = orders_delivered[orders_delivered['delivery_days'] < 0]
print("Заказов с отрицательным delivery_days:", len(anomaly))

Заказов с отрицательным delivery_days: 0


# месяц покупки (для когорт)

In [28]:
orders_delivered['order_month'] = (
    orders_delivered['order_purchase_timestamp']
    .dt.to_period('M')
)

print("Заказов по месяцам:")
print(orders_delivered['order_month'].value_counts().sort_index().head(10))
print("...")
print(orders_delivered['order_month'].value_counts().sort_index().tail(5))

Заказов по месяцам:
order_month
2016-09       1
2016-10     265
2016-12       1
2017-01     750
2017-02    1653
2017-03    2546
2017-04    2303
2017-05    3545
2017-06    3135
2017-07    3872
Freq: M, Name: count, dtype: int64
...
order_month
2018-04    6798
2018-05    6749
2018-06    6096
2018-07    6156
2018-08    6351
Freq: M, Name: count, dtype: int64


# оценки (H2)

In [29]:
# Убираем дубли: если на заказ несколько отзывов, берём последний
reviews_clean = (
    reviews[['order_id', 'review_score']]
    .drop_duplicates(subset='order_id', keep='last')
)

print("Отзывов до:", len(reviews))
print("Отзывов после удаления дублей:", len(reviews_clean))
print(f"Удалено дублей: {len(reviews) - len(reviews_clean)}")

# Присоединяем к orders_delivered
orders_delivered = orders_delivered.merge(
    reviews_clean,
    on='order_id',
    how='left'
)

print()
print("Пропусков в review_score:", orders_delivered['review_score'].isnull().sum())

Отзывов до: 99224
Отзывов после удаления дублей: 98673
Удалено дублей: 551

Пропусков в review_score: 646


# категории товаров (H3)
Категории лежат в products (португальские) + translation (английские). Но есть проблема: в одном заказе может быть несколько товаров из разных категорий.
Берём основную категорию — ту, которая принесла больше всего выручки в заказе

In [30]:
#присоединяем категории к товарам заказа
items_with_category = order_items.merge(
    products[['product_id', 'product_category_name']],
    on='product_id',
    how='left'
).merge(
    translation,
    on='product_category_name',
    how='left'
)

print("Пропусков в категории после merge:", 
      items_with_category['product_category_name_english'].isnull().sum())

Пропусков в категории после merge: 1627


In [31]:
items_with_category = order_items.merge(
    products[['product_id', 'product_category_name']],
    on='product_id',
    how='left'
).merge(
    translation,
    on='product_category_name',
    how='left'
)

print("Пропусков в категории после merge:", 
      items_with_category['product_category_name_english'].isnull().sum())

Пропусков в категории после merge: 1627


In [32]:
print("Всего строк:", len(items_with_category))
print("Пропусков в product_category_name_english:", 
      items_with_category['product_category_name_english'].isnull().sum())
print(f"Доля пропусков: {items_with_category['product_category_name_english'].isnull().mean():.2%}")
print()
print("Первые строки:")
print(items_with_category[['order_id', 'product_id', 'product_category_name', 'product_category_name_english', 'price']].head(10))

Всего строк: 112650
Пропусков в product_category_name_english: 1627
Доля пропусков: 1.44%

Первые строки:
                           order_id                        product_id  product_category_name product_category_name_english   price
0  00010242fe8c5a6d1ba2dd792cb16214  4244733e06e7ecb4970a6e2683c13e61             cool_stuff                    cool_stuff   58.90
1  00018f77f2f0320c557190d7a144bdd3  e5f2d52b802189ee658865ca93d83a8f               pet_shop                      pet_shop  239.90
2  000229ec398224ef6ca0657da4fc703e  c777355d18b72b67abbeef9df44fd0fd       moveis_decoracao               furniture_decor  199.00
3  00024acbcdf0a6daa1e931b038114c75  7634da152a4610f1595efa32f14722fc             perfumaria                     perfumery   12.99
4  00042b26cf59d7ce69dfabb4e55b4fd9  ac6c3623068f30de03045865e4e10089     ferramentas_jardim                  garden_tools  199.90
5  00048cc3ae777c65dbb7d2a0634bc1ea  ef92defde845ab8450f9d70c526ef70f  utilidades_domesticas                

# Определения главное категории

In [33]:
# Шаг 10.1: присоединяем категории к товарам
items_with_category = order_items.merge(
    products[['product_id', 'product_category_name']],
    on='product_id',
    how='left'
).merge(
    translation,
    on='product_category_name',
    how='left'
)

print("Всего строк:", len(items_with_category))
print("Пропусков в product_category_name_english:", 
      items_with_category['product_category_name_english'].isnull().sum())

Всего строк: 112650
Пропусков в product_category_name_english: 1627


# С максимальной выручкой:

In [34]:
# Шаг 10.2: разбираемся, откуда пропуски
no_portuguese = items_with_category[items_with_category['product_category_name'].isnull()]
no_english = items_with_category[
    items_with_category['product_category_name'].notnull() & 
    items_with_category['product_category_name_english'].isnull()
]

print("Без португальской категории:", len(no_portuguese))
print("С португальской, но без английской:", len(no_english))
if len(no_english) > 0:
    print("\nКатегории без перевода:")
    print(no_english['product_category_name'].value_counts())

Без португальской категории: 1603
С португальской, но без английской: 24

Категории без перевода:
product_category_name
portateis_cozinha_e_preparadores_de_alimentos    15
pc_gamer                                          9
Name: count, dtype: int64


In [35]:
# Шаг 10.3: заполняем пропуски ДО группировки (КРИТИЧНО!)
items_with_category['category_final'] = (
    items_with_category['product_category_name_english']
    .fillna(items_with_category['product_category_name'])
    .fillna('unknown')
)

print("Пропусков в category_final:", items_with_category['category_final'].isnull().sum())

Пропусков в category_final: 0


In [36]:
# Шаг 10.4: считаем выручку по (order_id, category_final)
category_revenue = (
    items_with_category
    .groupby(['order_id', 'category_final'])['price']
    .sum()
    .reset_index()
)

print("Строк в category_revenue:", len(category_revenue))
print("Уникальных order_id:", category_revenue['order_id'].nunique())
print()
print(category_revenue.head(10))

Строк в category_revenue: 99470
Уникальных order_id: 98666

                           order_id   category_final   price
0  00010242fe8c5a6d1ba2dd792cb16214       cool_stuff   58.90
1  00018f77f2f0320c557190d7a144bdd3         pet_shop  239.90
2  000229ec398224ef6ca0657da4fc703e  furniture_decor  199.00
3  00024acbcdf0a6daa1e931b038114c75        perfumery   12.99
4  00042b26cf59d7ce69dfabb4e55b4fd9     garden_tools  199.90
5  00048cc3ae777c65dbb7d2a0634bc1ea       housewares   21.90
6  00054e8431b9d7675808bcb819fb4a32        telephony   19.90
7  000576fe39319847cbb9d288c5617fa6     garden_tools  810.00
8  0005a1a1728c9d785b8e2b08b904576c    health_beauty  145.95
9  0005f50442cb953dcd1d21e1fb923495  books_technical   53.99


In [37]:
# Шаг 10.5: выбираем главную категорию для каждого заказа
main_category = (
    category_revenue
    .sort_values('price', ascending=False)
    .drop_duplicates(subset='order_id', keep='first')
    [['order_id', 'category_final']]
    .rename(columns={'category_final': 'main_category'})
)

print("Заказов в main_category:", len(main_category))
print("Уникальных категорий:", main_category['main_category'].nunique())
print()
print(main_category.head())

Заказов в main_category: 98666
Уникальных категорий: 74

                               order_id    main_category
1464   03caa2c082116e1d31e67e9ae3700499  fixed_telephony
44821  736e1922ae60d0d6a89247b851902527  fixed_telephony
3147   0812eb902a67711a1cb742b3cdaa65ae       housewares
99098  fefacc66af859508bf1a7934eab1e97f        computers
95215  f5136e38d1a14a4dbd87dff67da82701              art


In [38]:
# Шаг 10.6: присоединяем main_category к orders_delivered
orders_delivered = orders_delivered.merge(
    main_category,
    on='order_id',
    how='left'
)

print("Пропусков в main_category:", 
      orders_delivered['main_category'].isnull().sum())

Пропусков в main_category: 0


In [39]:
# Шаг 10.7: если пропуски есть — разбираемся
no_cat = orders_delivered[orders_delivered['main_category'].isnull()]
print("Заказов без main_category:", len(no_cat))

if len(no_cat) > 0:
    no_cat_ids = no_cat['order_id'].tolist()
    in_items = order_items[order_items['order_id'].isin(no_cat_ids)]
    print("Из них есть в order_items:", in_items['order_id'].nunique())
    
    # Если заказы без товаров — удаляем
    # (они всё равно не имеют revenue после следующего шага)

Заказов без main_category: 0


# Мастер таблица

In [40]:
master = orders_delivered.merge(
    items_agg,
    on='order_id',
    how='left'
).merge(
    payments_agg,
    on='order_id',
    how='left'
)

print("Размер мастер-таблицы:", master.shape)
print()
print("Пропуски в ключевых колонках:")
print(master[['revenue', 'freight', 'payment_value', 'main_category']].isnull().sum())

Размер мастер-таблицы: (96470, 18)

Пропуски в ключевых колонках:
revenue          0
freight          0
payment_value    1
main_category    0
dtype: int64


# Отбор колонок

In [41]:
columns_to_keep = [
    'order_id',
    'customer_unique_id',
    'customer_state',
    'order_purchase_timestamp',
    'order_month',
    'revenue',
    'freight',
    'payment_value',
    'review_score',
    'delivery_days',
    'is_late',
    'main_category'
]

missing = [col for col in columns_to_keep if col not in master.columns]
if missing:
    print("❌ Отсутствуют:", missing)
else:
    print("✅ Все колонки на месте")
    master = master[columns_to_keep].copy()
    print("Размер:", master.shape)

✅ Все колонки на месте
Размер: (96470, 12)


In [42]:
print("=== ФИНАЛЬНАЯ ПРОВЕРКА ===\n")

print(f"Всего заказов: {len(master):,}")
print(f"Уникальных клиентов: {master['customer_unique_id'].nunique():,}")
print(f"Период: {master['order_purchase_timestamp'].min().date()} — {master['order_purchase_timestamp'].max().date()}")
print()
print(f"Общая выручка: {master['revenue'].sum():,.2f}")
print(f"Средний чек: {master['revenue'].mean():,.2f}")
print(f"Средняя оценка: {master['review_score'].mean():.2f}")
print(f"Пропусков в review_score: {master['review_score'].isnull().sum():,}")
print(f"Средняя доставка: {master['delivery_days'].mean():.1f} дней")
print(f"Доля опозданий: {master['is_late'].mean():.1%}")
print(f"Уникальных категорий: {master['main_category'].nunique()}")
print(f"Пропусков в main_category: {master['main_category'].isnull().sum()}")

=== ФИНАЛЬНАЯ ПРОВЕРКА ===

Всего заказов: 96,470
Уникальных клиентов: 93,350
Период: 2016-09-15 — 2018-08-29

Общая выручка: 13,220,248.93
Средний чек: 137.04
Средняя оценка: 4.16
Пропусков в review_score: 646
Средняя доставка: 12.1 дней
Доля опозданий: 8.1%
Уникальных категорий: 74
Пропусков в main_category: 0


# Первые гипотезы

In [43]:
customer_late = master.groupby('customer_unique_id')['is_late'].max()
customer_orders = master.groupby('customer_unique_id').size()
returned = customer_orders >= 2

print("H1: Опоздания и возвращаемость")
print(f"  Клиентов: {len(customer_orders):,}")
print(f"  Вернулись: {returned.sum():,} ({returned.mean():.1%})")
print()
print(f"  Без опозданий: {returned[~customer_late].mean():.1%}")
print(f"  С опозданием:  {returned[customer_late].mean():.1%}")

print()
customer_score = master.groupby('customer_unique_id')['review_score'].min()
print("H2: Оценки и возвращаемость")
for score in [1, 2, 3, 4, 5]:
    mask = customer_score == score
    if mask.sum() > 0:
        print(f"  Мин. оценка {score}: вернулись {returned[mask].mean():.1%} (n={mask.sum():,})")

H1: Опоздания и возвращаемость
  Клиентов: 93,350
  Вернулись: 2,801 (3.0%)

  Без опозданий: 2.9%
  С опозданием:  4.6%

H2: Оценки и возвращаемость
  Мин. оценка 1: вернулись 4.5% (n=9,231)
  Мин. оценка 2: вернулись 4.1% (n=2,871)
  Мин. оценка 3: вернулись 4.1% (n=7,763)
  Мин. оценка 4: вернулись 3.0% (n=18,413)
  Мин. оценка 5: вернулись 2.5% (n=54,469)


In [44]:
master.to_csv('../data/processed_data/master_orders.csv', index=False)

print("Сохранено:", os.path.exists('../data/processed/master_orders.csv'))
print("Размер:", round(os.path.getsize('../data/processed_data/master_orders.csv') / 1024 / 1024, 2), "МБ")

Сохранено: False
Размер: 13.09 МБ


In [47]:
# Для дашборда: cohort_month и month_index
first_purchase = (
    master.groupby('customer_unique_id')['order_purchase_timestamp']
    .min().reset_index()
)
first_purchase.columns = ['customer_unique_id', 'first_purchase_date']
first_purchase['cohort_month'] = (
    first_purchase['first_purchase_date'].dt.to_period('M').astype(str)
)

master = master.merge(
    first_purchase[['customer_unique_id', 'cohort_month']],
    on='customer_unique_id',
    how='left'
)

# Считаем month_index от cohort_month
master['order_month_period'] = master['order_purchase_timestamp'].dt.to_period('M')
master['cohort_period'] = pd.PeriodIndex(master['cohort_month'], freq='M')
master['month_index'] = (
    master['order_month_period'] - master['cohort_period']
).apply(lambda x: x.n)

master.to_csv('../data/processed_data/master_for_bi.csv', index=False)
print("Добавлены cohort_month и month_index")

Добавлены cohort_month и month_index


In [49]:
import pandas as pd

check = pd.read_csv('../data/processed_data/master_for_bi.csv', nrows=1)
print("Колонки:", check.columns.tolist())
print()
print("Есть ли order_id:", 'order_id' in check.columns)

Колонки: ['order_id', 'customer_unique_id', 'customer_state', 'order_purchase_timestamp', 'order_month', 'revenue', 'freight', 'payment_value', 'review_score', 'delivery_days', 'is_late', 'main_category', 'cohort_month_x', 'order_month_period', 'cohort_period', 'month_index', 'cohort_month_y', 'cohort_month']

Есть ли order_id: True


In [51]:
import pandas as pd
check = pd.read_csv('../data/processed_data/master_for_bi.csv', nrows=1)
print("Колонки:", check.columns.tolist())
print("Есть ли order_id:", 'order_id' in check.columns)

Колонки: ['order_id', 'customer_unique_id', 'customer_state', 'order_purchase_timestamp', 'order_month', 'revenue', 'freight', 'payment_value', 'review_score', 'delivery_days', 'is_late', 'main_category', 'cohort_month_x', 'order_month_period', 'cohort_period', 'month_index', 'cohort_month_y', 'cohort_month']
Есть ли order_id: True


In [52]:
# Минимальная оценка клиента
customer_score = master.groupby('customer_unique_id')['review_score'].min().reset_index()
customer_score.columns = ['customer_unique_id', 'min_review_score']

master = master.merge(customer_score, on='customer_unique_id', how='left')

master.to_csv('../data/processed_data/master_for_bi.csv', index=False)
print("Добавлена колонка min_review_score")

Добавлена колонка min_review_score


In [54]:
import pandas as pd

check = pd.read_csv('../data/processed_data/master_for_bi.csv', nrows=5)
print("Колонки:", check.columns.tolist())
print()
print("Количество колонок:", len(check.columns))
print()
print(check.head())

Колонки: ['order_id', 'customer_unique_id', 'customer_state', 'order_purchase_timestamp', 'order_month', 'revenue', 'freight', 'payment_value', 'review_score', 'delivery_days', 'is_late', 'main_category', 'cohort_month_x', 'order_month_period', 'cohort_period', 'month_index', 'cohort_month_y', 'cohort_month', 'min_review_score']

Количество колонок: 19

                           order_id                customer_unique_id customer_state order_purchase_timestamp order_month  revenue  freight  payment_value  review_score  delivery_days  is_late  \
0  e481f51cbdc54678b7cc49136f2d6af7  7c396fd4830fd04220f754e42b4e5bff             SP      2017-10-02 10:56:33     2017-10    29.99     8.72          38.71           4.0              8    False   
1  53cdb2fc8bc7dce0b6741e2150273451  af07308b275d755c9edb36a90c618231             BA      2018-07-24 20:41:37     2018-07   118.70    22.76         141.46           4.0             13    False   
2  47770eb9100c2d0c44946d9cf07ec65d  3a653a41f6f9fc3d2a1

In [56]:
import pandas as pd

check = pd.read_csv('../data/processed_data/master_for_bi.csv', nrows=1)
print("Колонки:", check.columns.tolist())
print()
print("Есть ли is_returning:", 'is_returning' in check.columns)
print("Есть ли is_late:", 'is_late' in check.columns)
print("Есть ли order_id:", 'order_id' in check.columns)

Колонки: ['order_id', 'customer_unique_id', 'customer_state', 'order_purchase_timestamp', 'order_month', 'revenue', 'freight', 'payment_value', 'review_score', 'delivery_days', 'is_late', 'main_category', 'cohort_month_x', 'order_month_period', 'cohort_period', 'month_index', 'cohort_month_y', 'cohort_month', 'min_review_score']

Есть ли is_returning: False
Есть ли is_late: True
Есть ли order_id: True


In [58]:
import pandas as pd

# Загружаем мастер-таблицу
master = pd.read_csv(
    '../data/processed_data/master_orders.csv',
    parse_dates=['order_purchase_timestamp']
)

# is_returning: вернулся ли клиент (2+ заказа)
customer_orders = master.groupby('customer_unique_id').size()
master['is_returning'] = master['customer_unique_id'].map(customer_orders >= 2)

# order_month_str: месяц в формате YYYY-MM
master['order_month_str'] = master['order_purchase_timestamp'].dt.strftime('%Y-%m')

# order_year: год
master['order_year'] = master['order_purchase_timestamp'].dt.year

# is_late_str: строковая версия
master['is_late_str'] = master['is_late'].map({True: 'С опозданием', False: 'Без опозданий'})

# Сохраняем
master.to_csv('../data/processed_data/master_for_bi.csv', index=False)

print("Сохранено. Колонки:")
print(master.columns.tolist())
print()
print("is_returning есть:", 'is_returning' in master.columns)
print("is_late_str есть:", 'is_late_str' in master.columns)
print("order_month_str есть:", 'order_month_str' in master.columns)

Сохранено. Колонки:
['order_id', 'customer_unique_id', 'customer_state', 'order_purchase_timestamp', 'order_month', 'revenue', 'freight', 'payment_value', 'review_score', 'delivery_days', 'is_late', 'main_category', 'is_returning', 'order_month_str', 'order_year', 'is_late_str']

is_returning есть: True
is_late_str есть: True
order_month_str есть: True


In [64]:
import pandas as pd

# Загружаем ЧИСТУЮ мастер-таблицу (без cohort_month и прочего мусора)
master = pd.read_csv(
    '../data/processed_data/master_orders.csv',
    parse_dates=['order_purchase_timestamp']
)

print("Исходные колонки:", master.columns.tolist())

# === ДОБАВЛЯЕМ НУЖНЫЕ КОЛОНКИ ===

# 1. is_returning
customer_orders = master.groupby('customer_unique_id').size()
master['is_returning'] = master['customer_unique_id'].map(customer_orders >= 2)

# 2. order_month_str
master['order_month_str'] = master['order_purchase_timestamp'].dt.strftime('%Y-%m')

# 3. order_year
master['order_year'] = master['order_purchase_timestamp'].dt.year

# 4. is_late_str
master['is_late_str'] = master['is_late'].map({True: 'С опозданием', False: 'Без опозданий'})

# 5. min_review_score
customer_score = master.groupby('customer_unique_id')['review_score'].min().reset_index()
customer_score.columns = ['customer_unique_id', 'min_review_score']
master = master.merge(customer_score, on='customer_unique_id', how='left')

# 6. first_category
first_order = (
    master
    .sort_values('order_purchase_timestamp')
    .groupby('customer_unique_id')
    .first()
    .reset_index()
)
first_category = first_order[['customer_unique_id', 'main_category']].rename(
    columns={'main_category': 'first_category'}
)
master = master.merge(first_category, on='customer_unique_id', how='left')

# === СОХРАНЯЕМ ===

master.to_csv('../data/processed_data/master_for_bi.csv', index=False)

print()
print("Сохранено. Колонки:")
print(master.columns.tolist())
print()
print("Всего колонок:", len(master.columns))
print()
print("Проверка ключевых колонок:")
for col in ['order_id', 'customer_unique_id', 'revenue', 'payment_value',
            'review_score', 'is_late', 'is_returning', 'delivery_days',
            'main_category', 'min_review_score', 'first_category',
            'order_month_str', 'order_year', 'is_late_str']:
    print(f"  {col}: {'✅' if col in master.columns else '❌'}")

Исходные колонки: ['order_id', 'customer_unique_id', 'customer_state', 'order_purchase_timestamp', 'order_month', 'revenue', 'freight', 'payment_value', 'review_score', 'delivery_days', 'is_late', 'main_category']

Сохранено. Колонки:
['order_id', 'customer_unique_id', 'customer_state', 'order_purchase_timestamp', 'order_month', 'revenue', 'freight', 'payment_value', 'review_score', 'delivery_days', 'is_late', 'main_category', 'is_returning', 'order_month_str', 'order_year', 'is_late_str', 'min_review_score', 'first_category']

Всего колонок: 18

Проверка ключевых колонок:
  order_id: ✅
  customer_unique_id: ✅
  revenue: ✅
  payment_value: ✅
  review_score: ✅
  is_late: ✅
  is_returning: ✅
  delivery_days: ✅
  main_category: ✅
  min_review_score: ✅
  first_category: ✅
  order_month_str: ✅
  order_year: ✅
  is_late_str: ✅


In [67]:
import pandas as pd

# Загружаем чистую мастер-таблицу
master = pd.read_csv(
    '../data/processed_data/master_orders.csv',
    parse_dates=['order_purchase_timestamp']
)

print("Исходные колонки:", master.columns.tolist())
print()

# === ДОБАВЛЯЕМ ВСЕ КОЛОНКИ ДЛЯ BI ===

# 1. is_returning
customer_orders = master.groupby('customer_unique_id').size()
master['is_returning'] = master['customer_unique_id'].map(customer_orders >= 2)

# 2. order_month_str
master['order_month_str'] = master['order_purchase_timestamp'].dt.strftime('%Y-%m')

# 3. order_year
master['order_year'] = master['order_purchase_timestamp'].dt.year

# 4. is_late_str
master['is_late_str'] = master['is_late'].map({True: 'С опозданием', False: 'Без опозданий'})

# 5. min_review_score ← ВОТ ОНА
customer_score = master.groupby('customer_unique_id')['review_score'].min().reset_index()
customer_score.columns = ['customer_unique_id', 'min_review_score']
master = master.merge(customer_score, on='customer_unique_id', how='left')

# 6. customer_ever_late_str — было ли у клиента ХОТЯ БЫ ОДНО опоздание
customer_late = master.groupby('customer_unique_id')['is_late'].max().reset_index()
customer_late.columns = ['customer_unique_id', 'customer_ever_late']
master = master.merge(customer_late, on='customer_unique_id', how='left')
master['customer_ever_late_str'] = master['customer_ever_late'].map(
    {True: 'С опозданием', False: 'Без опозданий'}
)

# 7. first_category — категория ПЕРВОГО заказа
first_order = (
    master
    .sort_values('order_purchase_timestamp')
    .groupby('customer_unique_id')
    .first()
    .reset_index()
)
first_category = first_order[['customer_unique_id', 'main_category']].rename(
    columns={'main_category': 'first_category'}
)
master = master.merge(first_category, on='customer_unique_id', how='left')

# === СОХРАНЯЕМ ===

master.to_csv('../data/processed_data/master_for_bi.csv', index=False)

print("Сохранено. Колонки:")
print(master.columns.tolist())
print()
print("Проверка ключевых колонок:")
for col in ['order_id', 'customer_unique_id', 'revenue', 'payment_value',
            'review_score', 'is_late', 'is_returning', 'delivery_days',
            'main_category', 'min_review_score', 'first_category',
            'customer_ever_late_str', 'order_month_str', 'order_year',
            'is_late_str']:
    print(f"  {col}: {'✅' if col in master.columns else '❌'}")

Исходные колонки: ['order_id', 'customer_unique_id', 'customer_state', 'order_purchase_timestamp', 'order_month', 'revenue', 'freight', 'payment_value', 'review_score', 'delivery_days', 'is_late', 'main_category']

Сохранено. Колонки:
['order_id', 'customer_unique_id', 'customer_state', 'order_purchase_timestamp', 'order_month', 'revenue', 'freight', 'payment_value', 'review_score', 'delivery_days', 'is_late', 'main_category', 'is_returning', 'order_month_str', 'order_year', 'is_late_str', 'min_review_score', 'customer_ever_late', 'customer_ever_late_str', 'first_category']

Проверка ключевых колонок:
  order_id: ✅
  customer_unique_id: ✅
  revenue: ✅
  payment_value: ✅
  review_score: ✅
  is_late: ✅
  is_returning: ✅
  delivery_days: ✅
  main_category: ✅
  min_review_score: ✅
  first_category: ✅
  customer_ever_late_str: ✅
  order_month_str: ✅
  order_year: ✅
  is_late_str: ✅


In [69]:
import pandas as pd

check = pd.read_csv('../data/processed_data/master_for_bi.csv', nrows=1)
print("Колонки:", check.columns.tolist())
print()
print("Есть ли first_category:", 'first_category' in check.columns)
print("Есть ли min_review_score:", 'min_review_score' in check.columns)
print("Есть ли customer_ever_late_str:", 'customer_ever_late_str' in check.columns)

Колонки: ['order_id', 'customer_unique_id', 'customer_state', 'order_purchase_timestamp', 'order_month', 'revenue', 'freight', 'payment_value', 'review_score', 'delivery_days', 'is_late', 'main_category', 'is_returning', 'order_month_str', 'order_year', 'is_late_str', 'min_review_score', 'customer_ever_late', 'customer_ever_late_str', 'first_category']

Есть ли first_category: True
Есть ли min_review_score: True
Есть ли customer_ever_late_str: True
